# Lesson 29 — Reformulations and Symmetry

## Learning objectives

1. Recognize symmetric MILPs and the cost they impose on B&B.
2. Apply orbital, lex, and isomorphism-pruning techniques.
3. Recognize convexification reformulations: extended formulations, perspective.
4. Use `discopt`'s reformulation infrastructure / `/reformulate` skill.

## 1. Symmetry hurts B&B

If two variables $x_i, x_j$ are interchangeable (the constraint matrix is invariant under their swap), B&B explores symmetric subtrees independently. Each leaves equivalent solutions multiple times in the tree {cite:p}`Margot2010`.

Examples: bin packing with identical bins; multi-machine scheduling with identical machines; coloring with interchangeable colors.

## 2. Symmetry breaking

- **Lex constraints:** $x_1 \preceq_{\text{lex}} x_2 \preceq \dots$ — small subset of tightenings; always valid.
- **Orbital fixing/branching** {cite:p}`Margot2010`: at each node, identify orbits of the symmetry group; fix all members of an orbit consistently.
- **Isomorphism pruning:** in heavily symmetric problems (e.g., graph coloring), prune nodes that are isomorphic to one already explored.

Modern solvers detect simple symmetries automatically; complex ones require user input.

## 3. Reformulation as a reasoning task

Three families of reformulations matter most:

- **Extended formulations:** add auxiliary variables to make the formulation tighter (smaller LP-relaxation gap). Classic example: SOS2 piecewise via $\lambda$-binaries vs SOS-native.
- **Perspective reformulation:** for $f(x) y$ with binary $y$, the convex hull is $y f(x/y)$ (perspective function). Strong but sometimes painful {cite:p}`Liberti2009`.
- **Convexification of bilinear terms:** McCormick (Lesson 16) is one option; if the bilinear term has special structure (e.g., $x_i x_j$ both binary), there may be a stronger formulation.

{cite:p}`Liberti2009` is a systematic catalogue.

## 4. discopt's reformulation pipeline

`discopt.reformulate` runs:

1. **Detect** symmetries via constraint-matrix automorphisms.
2. **Detect** weak big-M and replace with tight bounds.
3. **Detect** disjunctive structure and apply convex-hull reformulation when better.
4. **Detect** bilinear terms and apply McCormick or RLT.

Logs explain each transformation. Use the `/reformulate` Claude Code skill (in `.claude/commands/reformulate.md`) to get an LLM-assisted suggestion list — but always verify.

In [ ]:
import numpy as np, discopt as do

# A small symmetric MILP: assign 4 jobs to 3 identical machines, minimize makespan
m = do.Model("makespan")
n_jobs = 4; n_mach = 3
p = np.array([3, 5, 2, 4])           # processing times
x = m.add_variables((n_jobs, n_mach), vtype="binary")
C = m.add_variable(name="C")          # makespan
for j in range(n_jobs):
    m.add_constraint(x[j, :].sum() == 1)
for k in range(n_mach):
    m.add_constraint(p @ x[:, k] <= C)
m.minimize(C)

# Without and with symmetry breaking
r0 = m.solve(symmetry_breaking=False)
r1 = m.solve(symmetry_breaking=True)
print(f"no-symbreak nodes: {r0.nodes}, time: {r0.runtime:.3f}s")
print(f"sym-break    nodes: {r1.nodes}, time: {r1.runtime:.3f}s")


## 5. Reformulation as a research practice

When stuck on a slow MILP/MINLP:

1. Look at the LP relaxation gap. If big, formulation is weak.
2. Inspect big-M values; tighten via OBBT.
3. Check for symmetry; add lex constraints.
4. Look for disjunctive structure; consider convex-hull reformulation.
5. Look for bilinear/concave terms; apply tighter relaxation.

This loop is the difference between a 2-day solve and a 2-second solve.

## References
{cite:p}`Margot2010,Liberti2009,Williams2013`.